# Why Machine Learning Works: Generalization, Hoeffding, and the Complexity Tradeoff
### A Pokémon vs. Digimon Story

Suppose we want to tell **Pokémon** from **Digimon** sprites using a single number: the **edge count** `e` of the image (Digimon tend to have busier line work, so a higher `e`). We will:

1. Build a **one-parameter threshold classifier** `f_h(e)` and measure its error with the **0/1 loss**.
2. See the **generalization gap**: we only ever train on a *sample* `D_train`, yet we care about the error on the whole world `D_all`.
3. Bound that gap with the **union bound + Hoeffding inequality**, and turn it into a **sample-complexity** rule ("how much data do I need?").
4. End on the **bias–complexity tradeoff** (魚與熊掌): bigger hypothesis classes lower the error floor but widen the gap.

**Roadmap of the three interactive demos**

| Demo | Cells | Question it answers |
|------|-------|---------------------|
| A. Threshold & loss | C03–C08 | How does a single parameter `h` set the error? |
| B. Generalization gap | C09–C11 | Why can a great training score still generalize badly? |
| C. Bound & tradeoff | C12–C20 | How much data do I need, and how complex a model should I pick? |

Everything runs on a small **synthetic** dataset — no copyrighted sprites required — so the lecture's numbers are reproducible.


In [ ]:
# --- Environment setup -------------------------------------------------------
import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:  # matplotlib is a hard dependency
    raise ImportError(
        "matplotlib is required for this notebook. Install it with: pip install matplotlib"
    ) from exc

# Enable inline plotting when running inside IPython/Colab; harmless otherwise.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Global reproducibility seed shared by every code cell.
SEED = 42
rng = np.random.default_rng(SEED)

# Default figure styling.
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 110

# --- sanity checks -----------------------------------------------------------
assert isinstance(rng, np.random.Generator)
assert SEED == 42
# Show that a freshly seeded generator is reproducible WITHOUT consuming `rng`.
_probe = np.random.default_rng(SEED).random()
print(f"numpy {np.__version__} | seed={SEED} | reproducible probe draw = {_probe:.6f}")


## The edge-count feature `e`

Digimon artwork generally has **busier line work** than Pokémon, so an edge detector returns a **higher edge count**. We treat that single scalar `e` as our feature.

Rather than run edge detection on copyrighted sprites, we **simulate** the feature with two overlapping class-conditional distributions, tuned to match the lecture slides:

- **Pokémon** (`y=0`): `e ~ Normal(mean ≈ 4000, σ)`
- **Digimon** (`y=1`): `e ~ Normal(mean ≈ 5500, σ)`

The two bumps **overlap** in the middle — and that overlap is exactly where a single threshold will have to make hard choices. The next cell builds this synthetic *universe* `D_all`, and the cell after that draws its histogram so you can see the overlap band.

> We simulate (a) to avoid copyright issues on real sprites, and (b) so we have a known ground-truth `D_all` to compare every sample against — the whole point of studying generalization.


In [ ]:
# --- Define the population D_all --------------------------------------------
MU_POKEMON = 4000.0   # mean edge count for Pokemon (y=0)
MU_DIGIMON = 5500.0   # mean edge count for Digimon (y=1)
SIGMA = 900.0         # shared spread; tuned so anchors land near slide values
CLASS_PRIOR = 0.5     # P(y=1) = P(Digimon)
N_UNIVERSE = 40000    # size of the synthetic ground-truth universe


def sample_population(n, rng=rng):
    """Draw `n` labeled (e, y) examples from the two-class edge-count model.

    y=0 -> Pokemon ~ N(MU_POKEMON, SIGMA); y=1 -> Digimon ~ N(MU_DIGIMON, SIGMA).
    Returns (e, y) with e clipped to [0, 10000].
    """
    if not (isinstance(n, (int, np.integer)) and n > 0):
        raise ValueError("n must be a positive integer")
    y = (rng.random(n) < CLASS_PRIOR).astype(int)
    means = np.where(y == 1, MU_DIGIMON, MU_POKEMON)
    e = rng.normal(means, SIGMA)
    e = np.clip(e, 0.0, 10000.0)
    return e.astype(float), y.astype(int)


# Materialize D_all from a DEDICATED generator so the universe is fixed and
# independent of later consumption of the global `rng`.
e_all, y_all = sample_population(N_UNIVERSE, rng=np.random.default_rng(SEED))
D_all = (e_all, y_all)

# --- report + sanity checks --------------------------------------------------
print(f"D_all size: {N_UNIVERSE}")
print(f"  Pokemon (y=0): n={np.sum(y_all == 0):5d}  mean e = {e_all[y_all==0].mean():7.1f}")
print(f"  Digimon (y=1): n={np.sum(y_all == 1):5d}  mean e = {e_all[y_all==1].mean():7.1f}")

es, ys = sample_population(1000)
assert es.shape == (1000,) and ys.shape == (1000,)
assert set(np.unique(y_all)).issubset({0, 1})
assert abs(e_all[y_all == 0].mean() - MU_POKEMON) < 100
assert abs(e_all[y_all == 1].mean() - MU_DIGIMON) < 100
assert e_all.min() >= 0 and e_all.max() <= 10000


In [ ]:
# --- Visualize the class-conditional edge-count distributions ----------------
bin_edges = np.linspace(0, 10000, 80)

e_pkmn, e_digi = e_all[y_all == 0], e_all[y_all == 1]
assert e_pkmn.size > 0 and e_digi.size > 0, "both classes must be present"

fig, ax = plt.subplots()
if e_pkmn.size:
    ax.hist(e_pkmn, bins=bin_edges, alpha=0.55, label='Pokémon (y=0)', color='tab:blue')
else:
    print("Warning: no Pokémon samples to plot")
if e_digi.size:
    ax.hist(e_digi, bins=bin_edges, alpha=0.55, label='Digimon (y=1)', color='tab:red')
else:
    print("Warning: no Digimon samples to plot")

# Highlight the overlap band where the threshold will live.
ax.axvspan(MU_POKEMON, MU_DIGIMON, color='gray', alpha=0.12, label='overlap band')
ax.set_xlabel('edge count e')
ax.set_ylabel('count')
ax.set_title('Edge-count distribution over D_all  (peaks ≈ 4000 / 5500)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# (C08 turns this static picture into an interactive threshold explorer.)
print("Two peaks near 4000 and 5500 with an overlapping middle band — that overlap is the hard region.")


## The threshold classifier, the hypothesis class, and the loss

We commit to the simplest possible model — a **threshold** on `e`:

$$ f_h(e) = \begin{cases} \text{Digimon } (1) & \text{if } e \ge h \\ \text{Pokémon } (0) & \text{if } e < h \end{cases} $$

The single unknown parameter is the threshold `h`. The **hypothesis class** is the set of all such classifiers,

$$ \mathcal{H} = \{\, f_h : h \in \{1, 2, \dots, 10000\} \,\}, \qquad |\mathcal{H}| = 10000. $$

We score a hypothesis with the **0/1 loss** — the *error rate*, the fraction of mistakes:

$$ L(h, D) = \frac{1}{|D|} \sum_{(e,y)\in D} \mathbf{1}\big[\,f_h(e) \ne y\,\big]. $$

This is exactly the **3-step ML recipe**:

1. **Function with an unknown parameter** — `f_h`, parameter `h`.
2. **Loss** — `L(h, D)`, the 0/1 error rate.
3. **Optimization** — search `h` to minimize the loss.

Next we implement these three pieces and find the best threshold on the whole universe `D_all`.


In [ ]:
# --- Prediction, loss, and the optimizer ------------------------------------
def predict(e, h):
    """Predict 1 (Digimon) when e >= h, else 0 (Pokemon). Vectorized."""
    return (np.asarray(e) >= h).astype(int)


def zero_one_loss(h, e, y):
    """0/1 error rate of threshold h on dataset (e, y), a float in [0, 1]."""
    e = np.asarray(e)
    y = np.asarray(y)
    return float(np.mean(predict(e, h) != y))


def best_threshold(e, y, H):
    """Sweep candidate thresholds H; return (h_star, loss_star) at the first argmin."""
    H = np.asarray(H)
    if H.size == 0:
        raise ValueError("H (candidate thresholds) must be non-empty")
    e = np.asarray(e)
    y = np.asarray(y)
    if e.shape[0] != y.shape[0]:
        raise ValueError("e and y must have the same length")
    losses = np.array([zero_one_loss(h, e, y) for h in H])
    idx = int(np.argmin(losses))  # first minimal-loss h (deterministic tie-break)
    return float(H[idx]), float(losses[idx])


# Canonical hypothesis class used everywhere downstream.
H_FULL = np.arange(2000, 8001, 1)

h_all, L_all = best_threshold(e_all, y_all, H_FULL)
print(f"Best threshold on D_all:  h_all = {h_all:.0f}   L(h_all, D_all) = {L_all:.3f}")

# --- sanity checks (computation correctness, not brittle slide-number pins) --
assert predict([3000, 6000], 5000).tolist() == [0, 1]
assert 0.0 <= L_all <= 1.0
# the optimizer's loss must be no worse than any fixed threshold's loss
assert L_all <= zero_one_loss(5000, e_all, y_all) + 1e-9
assert zero_one_loss(h_all, e_all, y_all) == L_all
# Slide reference (NOT asserted: this synthetic run need not reproduce the
# lecture's exact figures; the *mechanism*, not the number, is the point).
print(f"(slide reference h*~4824, L*~0.28 on the lecture data; "
      f"this synthetic run: h*={h_all:.0f}, L*={L_all:.3f})")


In [ ]:
# --- Interactive threshold explorer -----------------------------------------
def draw_threshold(h):
    h = int(np.clip(h, 2000, 8000))  # stay within the plotted range
    fig, ax = plt.subplots()
    ax.hist(e_all[y_all == 0], bins=bin_edges, alpha=0.55, label='Pokémon (y=0)', color='tab:blue')
    ax.hist(e_all[y_all == 1], bins=bin_edges, alpha=0.55, label='Digimon (y=1)', color='tab:red')
    ax.axvline(h, color='black', lw=2, label=f'threshold h={h}')
    ax.axvline(h_all, color='green', lw=1.2, ls='--', alpha=0.7, label=f'h_all={h_all:.0f}')
    loss = zero_one_loss(h, e_all, y_all)
    ax.set_title(f'h = {h}    L(h, D_all) = {loss:.3f}')
    ax.set_xlabel('edge count e')
    ax.set_ylabel('count')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(draw_threshold,
             h=widgets.IntSlider(min=2000, max=8000, step=25, value=int(h_all)))
except ImportError:
    print("ipywidgets not available — rendering one static frame at h = h_all.")
    draw_threshold(int(h_all))

# headless sanity: the callback runs and the loss at h_all matches L_all.
assert callable(draw_threshold)
assert abs(zero_one_loss(int(h_all), e_all, y_all) - L_all) < 0.02


## We never see `D_all` — only a sample `D_train`

In real life the whole universe `D_all` is hidden. We get a **finite training sample** `D_train`, pick the threshold that looks best on it,

$$ h_{train} = \arg\min_h \; L(h, D_{train}), $$

but what we actually *care about* is the **true loss** `L(h_train, D_all)`. The difference

$$ \text{generalization gap} = L(h_{train}, D_{all}) - L(h_{train}, D_{train}) $$

is the whole story of overfitting.

**Lucky vs. unlucky samples.** Two different training draws (`train1`, `train2`) of the *same size* can tell very different stories: a **lucky** sample gives a `h_train` whose true loss is close to the train loss, while an **unlucky** sample gives a small train loss that *hides* a large true loss. The next cell makes this concrete with one lucky and one unlucky seed; the cell after that resamples many times to show the gap as a distribution.


In [ ]:
# --- Single-sample experiment: lucky vs. unlucky -----------------------------
N_DEMO = 200


def run_single_trial(N, seed):
    if N >= N_UNIVERSE:
        raise ValueError("training set must be a small sample of the universe")
    g = np.random.default_rng(seed)
    e_tr, y_tr = sample_population(N, rng=g)
    h_train, train_loss = best_threshold(e_tr, y_tr, H_FULL)
    true_loss = zero_one_loss(h_train, e_all, y_all)
    return {'seed': seed, 'N': N, 'h_train': h_train,
            'train_loss': train_loss, 'true_loss': true_loss,
            'gap': true_loss - train_loss}


# Scan a fixed seed range and pick the smallest- and largest-gap draws so the
# lucky/unlucky contrast is robust rather than hand-picked.
scan = [run_single_trial(N_DEMO, s) for s in range(0, 51)]
lucky_result = min(scan, key=lambda r: r['gap'])
unlucky_result = max(scan, key=lambda r: r['gap'])
LUCKY_SEED, UNLUCKY_SEED = lucky_result['seed'], unlucky_result['seed']

print(f"{'case':10s} {'seed':>4s} {'h_train':>8s} {'train_loss':>11s} {'true_loss':>10s} {'gap':>7s}")
for tag, r in [('LUCKY', lucky_result), ('UNLUCKY', unlucky_result)]:
    print(f"{tag:10s} {r['seed']:4d} {r['h_train']:8.0f} "
          f"{r['train_loss']:11.3f} {r['true_loss']:10.3f} {r['gap']:7.3f}")
print("\nNote: a small TRAIN loss can hide a much larger TRUE loss — that is overfitting.")

assert 0 <= lucky_result['train_loss'] <= 1 and 0 <= lucky_result['true_loss'] <= 1
assert 0 <= unlucky_result['train_loss'] <= 1 and 0 <= unlucky_result['true_loss'] <= 1
assert unlucky_result['gap'] >= lucky_result['gap']
assert lucky_result['N'] == N_DEMO == 200


In [ ]:
# --- Interactive resampling: the gap as a distribution -----------------------
def resample_experiment(N, trials):
    if N < 1 or trials < 1:
        raise ValueError("N and trials must each be >= 1")
    trials = min(trials, 500)
    train_losses, true_losses = [], []
    for t in range(trials):
        g = np.random.default_rng(1000 + t)
        e_tr, y_tr = sample_population(N, rng=g)
        if e_tr.size == 0:
            continue
        h_train, train_loss = best_threshold(e_tr, y_tr, H_FULL)
        train_losses.append(train_loss)
        true_losses.append(zero_one_loss(h_train, e_all, y_all))
    return np.array(train_losses), np.array(true_losses)


def plot_gap(N, trials):
    if trials > 500:
        print("trials capped at 500 to bound runtime.")
    tl, vl = resample_experiment(N, trials)
    fig, ax = plt.subplots()
    ax.scatter(tl, vl, alpha=0.5, s=22, color='tab:purple')
    lo = min(tl.min(), vl.min()) if tl.size else 0.0
    hi = max(tl.max(), vl.max()) if tl.size else 1.0
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='train = true (y = x)')
    mean_gap = float(np.mean(vl - tl)) if tl.size else 0.0
    ax.set_xlabel('train loss  L(h_train, D_train)')
    ax.set_ylabel('true loss  L(h_train, D_all)')
    ax.set_title(f'N={N}, trials={len(tl)}   mean gap = {mean_gap:.3f}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(plot_gap,
             N=widgets.IntSlider(min=50, max=2000, step=50, value=200),
             trials=widgets.IntSlider(min=20, max=500, step=20, value=100))
except ImportError:
    print("ipywidgets not available — rendering one static frame at N=200, trials=100.")
    plot_gap(200, 100)

# headless checks
tl, vl = resample_experiment(200, 30)
assert tl.shape == (30,) and vl.shape == (30,)
assert np.all((tl >= 0) & (tl <= 1)) and np.all((vl >= 0) & (vl <= 1))
g_small = np.mean(resample_experiment(50, 60)[1] - resample_experiment(50, 60)[0])
g_large = np.mean(resample_experiment(2000, 60)[1] - resample_experiment(2000, 60)[0])
print(f"mean gap: N=50 -> {g_small:.3f}   N=2000 -> {g_large:.3f}  (more data tightens the gap)")


## When is a training set "bad"? — Union bound + Hoeffding

Call a training set **bad** if there exists *some* hypothesis whose train loss misrepresents its true loss by more than a tolerance `ε`:

$$ D_{train}\text{ is bad} \iff \exists\, h\in\mathcal{H}: \; \big| L(h, D_{train}) - L(h, D_{all}) \big| > \varepsilon. $$

If even one `h` is misleading, the learner can be fooled. We bound the probability of this happening in two moves:

- **Hoeffding (per hypothesis).** For a *fixed* `h`, the train loss is an average of `N` independent 0/1 terms, so it concentrates around the true loss:
$$ P\big(|L(h, D_{train}) - L(h, D_{all})| > \varepsilon\big) \le 2\,e^{-2N\varepsilon^2}. $$

- **Union bound (over all `h`).** "Some `h` is bad" is the union of the per-`h` bad events, so we just add them up:
$$ P(D_{train}\text{ is bad}) \;\le\; \sum_{h\in\mathcal{H}} 2\,e^{-2N\varepsilon^2} \;=\; |\mathcal{H}|\cdot 2\,e^{-2N\varepsilon^2}. $$

Read off the three knobs: **`N`** (more data → exponentially smaller), **`ε`** (looser tolerance → smaller, via `ε²` in the exponent), and **`|H|`** (more hypotheses → linearly larger). The next cell computes this bound and checks it against the slide numbers.


In [ ]:
# --- The Hoeffding / union-bound probability ---------------------------------
def hoeffding_raw(H_size, N, eps):
    """Unclipped union-bound value |H| * 2 * exp(-2 N eps^2) (matches pre-clip slides)."""
    if not (eps > 0 and N > 0 and H_size >= 1):
        raise ValueError("require eps > 0, N > 0, H_size >= 1")
    return float(H_size * 2.0 * np.exp(-2.0 * N * eps ** 2))


def hoeffding_bound(H_size, N, eps):
    """Probability bound on a bad sample, clipped to [0, 1]."""
    return float(min(1.0, hoeffding_raw(H_size, N, eps)))


# Validate against the slide anchors for |H| = 10000, eps = 0.1.
anchors = {100: 2707.0, 500: 0.91, 1000: 4.0e-5}
print(f"{'N':>5s} {'clipped bound':>14s} {'raw value':>12s} {'slide anchor':>14s}")
for N, anchor in anchors.items():
    print(f"{N:5d} {hoeffding_bound(10000, N, 0.1):14.5f} "
          f"{hoeffding_raw(10000, N, 0.1):12.5f} {anchor:14.5f}")

assert hoeffding_bound(10000, 100, 0.1) == 1.0                       # vacuous, clipped
assert abs(hoeffding_raw(10000, 100, 0.1) - 2707) / 2707 < 0.05      # raw anchor
assert abs(hoeffding_bound(10000, 500, 0.1) - 0.91) < 0.05
assert hoeffding_bound(10000, 1000, 0.1) < 1e-3
print("\nThe bound collapses from vacuous (>1) at N=100 to negligible at N=1000.")


In [ ]:
# --- Interactive bound explorer ----------------------------------------------
def plot_bound(N_max, H_size, eps):
    eps = max(float(eps), 0.01)  # avoid a degenerate flat curve
    N_grid = np.arange(50, int(N_max) + 1, 10)
    curve = np.minimum(1.0, H_size * 2.0 * np.exp(-2.0 * N_grid * eps ** 2))

    fig, ax = plt.subplots()
    ax.plot(N_grid, curve, color='tab:green', lw=2,
            label=f'|H|={H_size}, ε={eps:.2f}')
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('training-set size N')
    ax.set_ylabel('P(bad sample) upper bound')
    ax.set_title('Union-bound + Hoeffding: bound vs. N')

    # Only annotate the slide anchors when the curve matches the slide setting.
    if H_size == 10000 and abs(eps - 0.1) < 1e-9:
        for N in (100, 500, 1000):
            yb = min(1.0, 10000 * 2.0 * np.exp(-2.0 * N * 0.1 ** 2))
            ax.scatter([N], [yb], color='black', zorder=5)
            ax.annotate(f'N={N}\n{yb:.3g}', (N, yb),
                        textcoords='offset points', xytext=(6, 8), fontsize=8)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(plot_bound,
             N_max=widgets.IntSlider(min=200, max=3000, step=100, value=1500),
             H_size=widgets.Dropdown(options=[100, 1000, 10000, 100000], value=10000),
             eps=widgets.FloatSlider(min=0.05, max=0.3, step=0.01, value=0.1))
except ImportError:
    print("ipywidgets not available — rendering one static frame.")
    plot_bound(1500, 10000, 0.1)

# headless checks
_grid = np.arange(50, 1501, 10)
_curve = np.minimum(1.0, 10000 * 2.0 * np.exp(-2.0 * _grid * 0.1 ** 2))
assert np.all(np.diff(_curve) <= 1e-12)  # non-increasing in N
assert callable(plot_bound)


In [ ]:
# --- Empirical P(bad) vs. the theoretical bound ------------------------------
H_eval = np.arange(2000, 8001, 100)  # coarse sup grid (approximates sup over H_FULL)


def empirical_bad_fraction(N, eps, trials, H_eval):
    if not (eps > 0):
        raise ValueError("eps must be positive")
    trials = min(trials, 300)
    bad = 0
    counted = 0
    true_losses = {h: zero_one_loss(h, e_all, y_all) for h in H_eval}
    for t in range(trials):
        e_tr, y_tr = sample_population(N, rng=np.random.default_rng(5000 + t))
        if e_tr.size == 0:
            continue
        counted += 1
        worst = max(abs(zero_one_loss(h, e_tr, y_tr) - true_losses[h]) for h in H_eval)
        if worst > eps:
            bad += 1
    return bad / counted if counted else 0.0


def plot_empirical_vs_theory(eps, trials):
    N_sweep = [50, 100, 200, 400, 800, 1200]
    emp = [empirical_bad_fraction(N, eps, trials, H_eval) for N in N_sweep]
    theo = [hoeffding_bound(len(H_FULL), N, eps) for N in N_sweep]

    fig, ax = plt.subplots()
    ax.plot(N_sweep, theo, 'o-', color='tab:green', label='Hoeffding bound (|H|=len(H_FULL))')
    ax.scatter(N_sweep, emp, color='tab:purple', zorder=5, label='empirical bad fraction')
    ax.set_xlabel('training-set size N')
    ax.set_ylabel('P(bad sample)')
    ax.set_title(f'Empirical vs. theory  (ε={eps}, trials={min(trials,300)})')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("H_eval is coarser than H_FULL, so the empirical sup is a LOWER estimate; "
          "the theoretical bound still sits above it.")


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(plot_empirical_vs_theory,
             eps=widgets.fixed(0.1),
             trials=widgets.IntSlider(min=30, max=300, step=30, value=100))
except ImportError:
    print("ipywidgets not available — rendering one static frame.")
    plot_empirical_vs_theory(0.1, 100)

# headless checks
f = empirical_bad_fraction(100, 0.1, 50, H_eval)
assert 0.0 <= f <= 1.0
for N in (50, 200, 800):
    assert empirical_bad_fraction(N, 0.1, 50, H_eval) <= hoeffding_bound(len(H_FULL), N, 0.1) + 0.1


## How much data do I need? — Sample complexity

Turn the bound around. We want the bad-sample probability to be at most some confidence budget `δ`:

$$ |\mathcal{H}| \cdot 2\,e^{-2N\varepsilon^2} \;\le\; \delta. $$

Solve for `N`. Take logs and rearrange:

$$ \boxed{\,N \;\ge\; \dfrac{\ln\!\big(2|\mathcal{H}|/\delta\big)}{2\varepsilon^2}\,} $$

This is the **sample-complexity** rule — a concrete **data budget**. It says we need *more* data when we want a tighter tolerance `ε` (note the `ε²`), a smaller failure probability `δ`, or a larger hypothesis class `|H|` (though only **logarithmically** in `|H|`, which is why finite `H` is so forgiving). The next cell computes it and confirms the slide's target of `N ≥ 610`.


In [ ]:
# --- Sample complexity: the data budget --------------------------------------
def sample_complexity(H_size, delta, eps):
    """Smallest N with |H| * 2 * exp(-2 N eps^2) <= delta, i.e. the boxed formula."""
    if not (0 < delta < 1 and eps > 0 and H_size >= 1):
        raise ValueError("require 0 < delta < 1, eps > 0, H_size >= 1")
    return int(np.ceil(np.log(2.0 * H_size / delta) / (2.0 * eps ** 2)))


N_required_target = sample_complexity(10000, 0.1, 0.1)
print(f"|H|=10000, delta=0.1, eps=0.1  ->  N >= {N_required_target}  (slide target ≈ 610)")


def show_budget(H_size, delta, eps):
    N = sample_complexity(H_size, delta, eps)
    print(f"With |H|={H_size}, δ={delta:.2f}, ε={eps:.2f}:  "
          f"you need at least N = {N} training examples.")


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(show_budget,
             H_size=widgets.Dropdown(options=[100, 1000, 10000, 100000], value=10000),
             delta=widgets.FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1),
             eps=widgets.FloatSlider(min=0.05, max=0.3, step=0.01, value=0.1))
except ImportError:
    print("ipywidgets not available — rendering one static readout.")
    show_budget(10000, 0.1, 0.1)

# headless checks
assert abs(sample_complexity(10000, 0.1, 0.1) - 610) <= 2
assert isinstance(sample_complexity(10000, 0.1, 0.1), int)
assert sample_complexity(10000, 0.05, 0.1) > sample_complexity(10000, 0.1, 0.1)   # smaller delta
assert sample_complexity(10000, 0.1, 0.05) > sample_complexity(10000, 0.1, 0.1)   # smaller eps
assert hoeffding_bound(10000, sample_complexity(10000, 0.1, 0.1), 0.1) <= 0.1     # inverted bound holds


## The bias–complexity tradeoff (魚與熊掌)

We can now read the expected true loss of `h_train` as a sum of two competing pieces:

$$ \underbrace{L(h_{train}, D_{all})}_{\text{what we pay}} \;\approx\; \underbrace{L(h_{all}^{\mathcal H}, D_{all})}_{\text{approximation floor}} \;+\; \underbrace{\text{generalization gap}}_{\text{estimation error}}. $$

- A **larger** `|H|` can fit the world better, so it **lowers the approximation floor** — but the union bound just told us a larger `|H|` **widens the gap**.
- A **smaller** `|H|` **narrows the gap** — but **raises the floor**, because the class may not contain a good classifier at all.

You cannot freely have both — fish and bear's paw, 魚與熊掌不可兼得. The total loss is therefore **U-shaped** in `|H|`: too simple underfits (high floor), too complex overfits (high gap), and the sweet spot is in between. **Can we have both?** Let's plot the U-curve and watch the optimum move as we add data.


In [ ]:
# --- Model the tradeoff: floor + gap across hypothesis classes ----------------
def build_class_family(granularities):
    """For each grid step, build a threshold class H = arange(2000, 8001, step)."""
    family = []
    span = 8000 - 2000
    for step in granularities:
        if step <= 0 or step > span:
            raise ValueError(f"invalid grid step {step}")
        family.append(np.arange(2000, 8001, step))
    return family


GRANULARITIES = [2000, 1000, 500, 200, 100, 50, 20, 10, 5, 1]


def tradeoff_curves(N, trials=60):
    family = build_class_family(GRANULARITIES)
    rows = []
    for H in family:
        floor = best_threshold(e_all, y_all, H)[1]  # best achievable TRUE loss in this class
        gaps = []
        for t in range(trials):
            g = np.random.default_rng(7000 + t)
            e_tr, y_tr = sample_population(N, rng=g)
            h_train = best_threshold(e_tr, y_tr, H)[0]
            gaps.append(zero_one_loss(h_train, e_all, y_all) - floor)
        gap = max(0.0, float(np.mean(gaps)))  # clip Monte-Carlo negatives
        rows.append((len(H), floor, gap))
    rows.sort(key=lambda r: r[0])  # ascending |H|
    h_sizes = np.array([r[0] for r in rows])
    floors = np.array([r[1] for r in rows])
    gaps = np.array([r[2] for r in rows])
    totals = floors + gaps
    return h_sizes, floors, gaps, totals


# Default curves at N=200 for the interactive plot in C20.
h_sizes, approximation_floor, gap, totals = tradeoff_curves(200)
print(f"{'|H|':>6s} {'floor':>7s} {'gap':>7s} {'total':>7s}")
for hs, fl, gp, tt in zip(h_sizes, approximation_floor, gap, totals):
    print(f"{hs:6d} {fl:7.3f} {gp:7.3f} {tt:7.3f}")

assert h_sizes.shape == approximation_floor.shape == gap.shape == totals.shape
assert np.all(np.diff(h_sizes) > 0)
assert approximation_floor[0] >= approximation_floor[-1] - 0.02
assert np.all(gap >= -1e-9)


In [ ]:
# --- Interactive U-curve: optimal complexity vs. data size -------------------
def plot_tradeoff(N):
    hs, floors, gaps, tots = tradeoff_curves(int(N))
    argmin = int(np.argmin(tots))
    optimal_h_size = hs[argmin]

    fig, ax = plt.subplots()
    ax.plot(hs, floors, 'o-', color='tab:blue', label='approximation floor (bias)')
    ax.plot(hs, gaps, 'o-', color='tab:red', label='generalization gap (variance)')
    ax.plot(hs, tots, 'o-', color='black', lw=2, label='expected true loss (total)')
    ax.axvline(optimal_h_size, color='gray', ls='--', alpha=0.8)
    ax.annotate('optimal complexity',
                (optimal_h_size, tots[argmin]),
                textcoords='offset points', xytext=(8, 12), fontsize=9)
    ax.set_xscale('log')
    ax.set_xlabel('|H|  (hypothesis-class size, log scale)')
    ax.set_ylabel('loss')
    ax.set_title(f'Bias–complexity tradeoff   N={N}   optimal |H|={optimal_h_size}')
    ax.legend()
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    from ipywidgets import interact
    interact(plot_tradeoff,
             N=widgets.IntSlider(min=50, max=2000, step=50, value=200))
except ImportError:
    print("ipywidgets not available — rendering one static frame at N=200.")
    plot_tradeoff(200)

print("As N grows, the gap term shrinks, so the optimal |H| shifts RIGHT — "
      "more data lets you afford a more complex model.")

# headless checks
_hs, _fl, _gp, _tot = tradeoff_curves(200)
_arg = int(np.argmin(_tot))
if _arg in (0, len(_tot) - 1):
    print("Notice: minimum at an endpoint for N=200 (Monte-Carlo noise); curve still U-leaning.")
assert callable(plot_tradeoff)


## Wrap-up: why machine learning works

We built one tiny model — a threshold on the edge count — and used it to see the whole theory of generalization:

- **Generalization gap (C09–C11).** We optimize on `D_train` but are judged on `D_all`. A small train loss can hide a large true loss; resampling showed the gap is a *distribution* that **tightens as `N` grows**.
- **Hoeffding + union bound (C12–C15).** `P(bad sample) ≤ |H| · 2·exp(−2Nε²)`. More data shrinks it exponentially; a bigger `H` inflates it only linearly. Monte-Carlo confirmed the bound is a valid (loose) envelope.
- **Sample complexity (C16–C17).** Inverting the bound gives `N ≥ ln(2|H|/δ)/(2ε²)` — a concrete data budget (here, `N ≥ 610`).
- **Bias–complexity tradeoff (C18–C20).** Total loss = approximation floor + gap. The floor falls with `|H|`, the gap rises, so the best complexity is a U-curve sweet spot that **moves right as you collect more data**.

**Why deep learning is exciting.** Modern models seem to *lower the approximation floor* (very expressive) **without paying the full estimation gap** the classic bound predicts — exactly the "can we have both?" question, and an active research frontier.

**Further reading / extensions**
- Replace 0/1 loss with **cross-entropy** for differentiable optimization.
- Beyond counting `|H|`: the **VC dimension** and Rademacher complexity bound *infinite* hypothesis classes.
- Try widening `SIGMA` or shifting the class means and re-running the demos to see the floor and gap respond.
